In [1]:

import os
import numpy as np
import cv2
np.random.seed(1337)
import gc

from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense, Conv2D,MaxPool2D,BatchNormalization,GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MultiLabelBinarizer

img_size = 128 # set the image size to 128
print(os.listdir())
dataset = os.listdir("music_dataset_spectro_full_single/train") #use as labels
labels = dataset
print(labels)

['.git', '.vscode', 'audio_separator_dataset', 'Checkpoint_tester.py', 'Classifiaction_model_full.keras', 'large separation dataset output', 'longSongSplitter.py', 'Mix_maker.py', 'model saves', 'model_classes_full.ipynb', 'model_classes_full_multi.ipynb', 'model_separation(large dataset).ipynb', 'model_separation(small dataset).ipynb', 'music_dataset_spectro_full_single', 'Music_mixes_dataset_classification', 'Music_separation_dataset_large', 'music_separation_dataset_small', 'my_separator_model_prototype.keras', 'prediction images', 'README.md', 'small separation dataset output', 'spectrogramMaker.py', 'spectrogramMakerMix.py', 'spectrogramMakerStem.py', 'wavs from spectrograms']
['Accordion', 'Acoustic_Guitar', 'Banjo', 'Bass_Guitar', 'Clarinet', 'cowbell', 'Dobro', 'Drum_set', 'Electric_Guitar', 'flute', 'Harmonium', 'Horn', 'Keyboard', 'Mandolin', 'Organ', 'Piano', 'Saxophone', 'Shakers', 'Tambourine', 'Trombone', 'Trumpet', 'Ukulele', 'vibraphone', 'Violin']


In [2]:
def get_dataset_array_train(data_dir):
    data = []
    path = os.path.join("Music_mixes_dataset_classification/train/"+"mix")
    for img in os.listdir("Music_mixes_dataset_classification/train/"+"mix"): #loop through mixes
        class_num = []
        for stem in os.listdir("Music_mixes_dataset_classification/train/"+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0] #split names so the instruments are used as labels
                class_num.append(labels.index(label))

        try:
            img_arr = cv2.imread(os.path.join(path,img),0)
            resized_arr =img_arr[:img_size,:img_size]
            data.append([resized_arr,class_num])
            gc.collect()
        except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [3]:
def get_dataset_array_test(data_dir):
    data = []
    path = os.path.join("Music_mixes_dataset_classification/test/"+"mix")
    for img in os.listdir("Music_mixes_dataset_classification/test/"+"mix"):#loop through mixes
        class_num = []
        for stem in os.listdir("Music_mixes_dataset_classification/test/"+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0]  #split names so the instruments are used as labels
                class_num.append(labels.index(label))

        try:
            img_arr = cv2.imread(os.path.join(path,img),0)
            resized_arr = img_arr[:img_size,:img_size]
            data.append([resized_arr,class_num])
            gc.collect()
        except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [4]:
def get_dataset_array_valid(data_dir):
    data = []
    path = os.path.join("Music_mixes_dataset_classification/valid/"+"mix")
    for img in os.listdir("Music_mixes_dataset_classification/valid/"+"mix"):#loop through mixes
        class_num = []
        for stem in os.listdir("Music_mixes_dataset_classification/valid/"+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0] #split names so the instruments are used as labels
                class_num.append(labels.index(label))

        try:
            img_arr = cv2.imread(os.path.join(path,img),0)
            resized_arr = img_arr[:img_size,:img_size]
            data.append([resized_arr,class_num])
            gc.collect()
        except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [5]:
train = get_dataset_array_train("music_dataset_spectro_full_single/train/")
test = get_dataset_array_test("music_dataset_spectro_full_single/test/")
valid = get_dataset_array_valid("music_dataset_spectro_full_single/valid/")

In [6]:
print(train.shape) # see the shapes
print(test.shape) # see the shapes
print(valid.shape) # see the shapes

(32833, 2)
(4105, 2)
(4103, 2)


In [7]:
x_train = []
y_train = []

x_val = []
y_val = []

x_test = []
y_test = []

for feature, label in train: #loop through the train array and split into x image and y labels
    x_train.append(feature)
    y_train.append(label)

del train

for feature, label in test:#loop through the test array and split into x image and y labels
    x_test.append(feature)
    y_test.append(label)
del test

for feature, label in valid:#loop through the valid array and split into x image and y labels
    x_val.append(feature)
    y_val.append(label)
     
del valid

In [8]:
gc.collect()
x_train = np.array(x_train)/255 #normalize the images
gc.collect()
x_test = np.array(x_test)/255 #normalize the images
gc.collect()
x_val = np.array(x_val)/255 #normalize the images
gc.collect()

0

In [9]:
x_train = x_train.reshape(-1, img_size, img_size, 1) # reshape images

mlb = MultiLabelBinarizer() # to make the labels multi hot

y_train = mlb.fit_transform(y_train) # make the labels multi hot

x_val = x_val.reshape(-1, img_size, img_size, 1) # reshape images

y_val = mlb.fit_transform(y_val) # make the labels multi hot


x_test = x_test.reshape(-1, img_size, img_size, 1) # reshape images

y_test = mlb.fit_transform(y_test)  # make the labels multi hot


In [29]:
# Model setup
model = Sequential()

#first block
model.add(Conv2D(32, (3,3), activation = 'relu', padding="same", input_shape = (img_size, img_size, 1))) # input the image shape
model.add(BatchNormalization())  # normalize image
model.add(Conv2D(32, (3,3), activation = 'relu', padding="same")) 
model.add(MaxPool2D((2,2))) # downsample image

#second block
model.add(Conv2D(64, (3,3), activation = 'relu', padding="same"))
model.add(BatchNormalization()) # normalize image
model.add(Conv2D(64, (3,3), activation = 'relu', padding="same"))
model.add(MaxPool2D((2,2))) # downsample image

#third block
model.add(Conv2D(128, (3,3), activation = 'relu', padding="same"))
model.add(BatchNormalization()) # normalize image
model.add(Conv2D(128, (3,3), activation = 'relu', padding="same"))
model.add(GlobalAveragePooling2D())

#last block
model.add(Dense(units = 512, activation = 'relu')) #dense layer
model.add(Dropout(0.4))
model.add(Dense(units = 24, activation = 'sigmoid'))

model.compile(
              optimizer = 'adam', loss = 'binary_crossentropy',
              metrics = ['binary_accuracy','accuracy']
              )
     

In [30]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 24)             │        12,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 365,688 (1.39 MB)

 Trainable params: 365,240 (1.39 MB)

 Non-trainable params: 448 (1.75 KB)

In [31]:
print(x_train.shape) 
print(y_train.shape)

(32833, 128, 128, 1)
(32833, 24)


In [32]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_binary_accuracy', patience = 2, verbose = 1, factor = 0.3, min_lr = 0.000001)
stop_early = EarlyStopping("val_binary_accuracy",patience = 3, verbose = 1)

In [33]:
batch_size = 32
n_epochs = 50
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = (x_val, y_val),
                    callbacks = [learning_rate_reduction,stop_early], shuffle = True)

Epoch 1/50
1027/1027 ━━━━━━━━━━━━━━━━━━━━ 455s 441ms/step - accuracy: 0.0364 - binary_accuracy: 0.8947 - loss: 0.3415 - val_accuracy: 0.0124 - val_binary_accuracy: 0.8958 - val_loss: 0.3353 - learning_rate: 0.0010
Epoch 2/50
1027/1027 ━━━━━━━━━━━━━━━━━━━━ 448s 436ms/step - accuracy: 0.0376 - binary_accuracy: 0.8953 - loss: 0.3374 - val_accuracy: 0.0171 - val_binary_accuracy: 0.8958 - val_loss: 0.3350 - learning_rate: 0.0010
Epoch 3/50
1026/1027 ━━━━━━━━━━━━━━━━━━━━ 0s 429ms/step - accuracy: 0.0374 - binary_accuracy: 0.8954 - loss: 0.3371
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
1027/1027 ━━━━━━━━━━━━━━━━━━━━ 452s 440ms/step - accuracy: 0.0351 - binary_accuracy: 0.8953 - loss: 0.3369 - val_accuracy: 0.0999 - val_binary_accuracy: 0.8958 - val_loss: 0.3349 - learning_rate: 0.0010
Epoch 4/50
1027/1027 ━━━━━━━━━━━━━━━━━━━━ 452s 440ms/step - accuracy: 0.0409 - binary_accuracy: 0.8953 - loss: 0.3364 - val_accuracy: 0.0188 - val_binary_accuracy: 0.8958 - val_

In [34]:
gc.collect()
print("loss of the model is - " , model.evaluate(x_test,y_test)[0])
print("Accuracy of the model is - " , model.evaluate(x_test,y_test)[1]*100 , "%")
model.save('Classifiaction_model_full_multi.keras')

129/129 ━━━━━━━━━━━━━━━━━━━━ 12s 94ms/step - accuracy: 0.0251 - binary_accuracy: 0.8957 - loss: 0.3345
loss of the model is -  0.3345317244529724
129/129 ━━━━━━━━━━━━━━━━━━━━ 12s 92ms/step - accuracy: 0.0251 - binary_accuracy: 0.8957 - loss: 0.3345
Accuracy of the model is -  89.57066535949707 %


In [21]:
predictions=model.predict(x_test)
pred_labels= np.where(predictions>0.5)

129/129 ━━━━━━━━━━━━━━━━━━━━ 12s 96ms/step


In [1]:
img = cv2.imread("music_dataset_spectro_full_single/test/Accordion/2864_Accordion.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
prediction = np.argmax(pred,1)
print(labels[prediction])


NameError: name 'cv2' is not defined

In [36]:
img = cv2.imread("Music_mixes_dataset_classification/test/mix/32836_mix.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
print(pred)
prediction = np.argmax(pred,1)
print(labels[prediction])

['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
[[0.09739322 0.10387162 0.09694903 0.10022099 0.10093796 0.09348806
  0.10065683 0.09759429 0.10458533 0.09822105 0.10772406 0.09831454
  0.09560847 0.09637304 0.11033848 0.10162962 0.10071411 0.10139136
  0.10126057 0.10233712 0.10564472 0.10194691 0.09971515 0.10841563]]
['Organ']


In [35]:
img = cv2.imread("Music_mixes_dataset_classification/test/mix/32844_mix.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
print(pred)
prediction = np.argmax(pred,1)
print(labels[prediction])

['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
[[0.10281377 0.10856538 0.10246617 0.104022   0.10693789 0.0972807
  0.10521085 0.10498907 0.11110108 0.10529784 0.11390807 0.10533275
  0.10044312 0.10065424 0.11675914 0.1072031  0.10596249 0.10764855
  0.10615558 0.10986157 0.11132622 0.10848486 0.10504122 0.1141059 ]]
['Organ']
